# Simulazione: GWB (13/3, -14) aggiunto alle simulazioni di band noise gia fatte

Invece di rigenerare tutto con un modello nuovo (che cambia la realizzazione di tutti i processi, band noise
compreso), il GWB viene **sommato** ai dati gia simulati:

- `sim_data_noise_bn_A125_sv44_50componenti/` (band noise a -12.5) + GWB  ->  `sim_data_noise_bn_A125_GWB_A140_sv44_50componenti/`
- `sim_data_noise_bn_OFF_sv44_50componenti/` (band noise spento) + GWB  ->  `sim_data_noise_bn_OFF_GWB_A140_sv44_50componenti/`

Cosi red noise, DM, scattering e band noise hanno **la stessa identica realizzazione** delle simulazioni OFF e A125
del notebook 21, e l unica differenza e il GWB.

La realizzazione del solo GWB si ottiene come differenza di due estrazioni con lo stesso seed e lo stesso modello:
una con GWB a -14 e una con GWB a -30 (spento), entrambe con il band noise spento.

In [ ]:
import numpy as np
import os, glob, json
import pickle

import sys

# this is the path to the discovery folder
sys.path.insert(0, "./")
import discovery_new as ds

import matplotlib.pyplot as plt
import scipy
import jax.numpy as jnp

from scipy.stats import uniform
import jax

import copy

In [ ]:
log10_A_gwb = -14.0

In [ ]:
# this are the data we want to replicate
path = './dati/DR2low_feathers/'
s_dsfiles = np.sort(os.listdir(path))
psr = [ds.pulsar.Pulsar.read_feather(path + f'{psrfile}') for psrfile in s_dsfiles][9]
print('pulsar :', psr.name)
print('N_TOA  :', len(psr.toas))

In [ ]:
# quante ToA prende la selezione LOFAR, e a che frequenze
sel = ds.lofar_backends(psr)
mask_lofar = np.any(np.vstack([np.asarray(m, dtype=bool) for m in sel.values()]), axis=0)
print('backend selezionati :', sorted(sel.keys()))
print(f'ToA nella selezione : {mask_lofar.sum()} su {len(psr.toas)}  ({100*mask_lofar.mean():.1f}%)')
print(f'frequenze           : {psr.freqs[mask_lofar].min():.1f} - {psr.freqs[mask_lofar].max():.1f} MHz')
print('componenti band noise: 50')

In [ ]:
###### Let's put in the model the components we want to simulate (comment a line if you don't want to inject that process)
## GWB + scattering con alpha = 4.4 + BAND NOISE sui backend LOFAR

# data
model = [psr.residuals]
# timing model
model += [ds.makegp_timing(psr, svd=True, variance=1e-40)]
# white noise parameters
model += [ds.makenoise_measurement(psr, psr.noisedict, tnequad = True, ecorr=True)]
# red noise paramters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_red_components'], T=ds.getspan(psr), name='red_noise' ))
# gwb paramters
model.append(ds.makegp_fourier(psr, ds.powerlaw, 20, T=ds.getspan(psr), name='gwb'))
# DM variation parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_dm_components'], T=ds.getspan(psr), name='dm_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 2.0, tndm = False)))
# scattering noise parameters  --> alpha = 4.4
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_chrom_components'], T=ds.getspan(psr), name='sv_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 4.4, tndm = False)))
# BAND NOISE sui backend LOFAR, 50 componenti
model.append(ds.makegp_fourier_select(psr, ds.powerlaw, components=50, T=ds.getspan(psr),
                                      name='band_noise', selection=ds.lofar_backends))

In [ ]:
gml = ds.PulsarLikelihood(model)
logL = ds.PulsarLikelihood(model).logL

In [ ]:
# tutti i valori standard, band noise spento; cambia solo l ampiezza del GWB
truths_gwb = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
    'J1640+2224_gwb_gamma': 13/3,
    'J1640+2224_gwb_log10_A': log10_A_gwb,
    'J1640+2224_band_noise_gamma': 3.0,
    'J1640+2224_band_noise_log10_A': -30.0,
}
truths_nogwb = dict(truths_gwb)
truths_nogwb['J1640+2224_gwb_log10_A'] = -30.0

assert set(logL.params) == set(truths_gwb.keys())

In [ ]:
# stesso seed: le due estrazioni differiscono solo per il GWB
_, res_gwb   = gml.sample(jax.random.key(1), truths_gwb)
_, res_nogwb = gml.sample(jax.random.key(1), truths_nogwb)

gwb_only = np.array(res_gwb) - np.array(res_nogwb)
print('rms del solo GWB:', np.std(gwb_only))

In [ ]:
# controllo: il GWB e acromatico, quindi ToA della stessa epoca a frequenze diverse hanno lo stesso ritardo
giorno = np.round(psr.toas / 86400)
variazione = [np.ptp(gwb_only[giorno == g]) for g in np.unique(giorno) if (giorno == g).sum() > 3]
print('max variazione del GWB entro la stessa epoca:', np.max(variazione), '(deve essere ~0)')

## Somma del GWB alle simulazioni di band noise

In [ ]:
for nome in ['A125', 'OFF']:
    base = ds.pulsar.Pulsar.read_feather(f'sim_data_noise_bn_{nome}_sv44_50componenti/' + psr.name)

    sim_psr = copy.deepcopy(base)
    sim_psr.residuals = np.array(base.residuals) + gwb_only

    dirpath = f'sim_data_noise_bn_{nome}_GWB_A140_sv44_50componenti/'
    if not os.path.exists(dirpath):
        os.makedirs(dirpath)
    ds.pulsar.Pulsar.save_feather(sim_psr, dirpath + psr.name, noisedict=psr.noisedict)
    print('salvato in', os.path.abspath(dirpath))

In [ ]:
# controlli: rileggo i feather appena scritti
for nome in ['A125', 'OFF']:
    base  = ds.pulsar.Pulsar.read_feather(f'sim_data_noise_bn_{nome}_sv44_50componenti/' + psr.name)
    check = ds.pulsar.Pulsar.read_feather(f'sim_data_noise_bn_{nome}_GWB_A140_sv44_50componenti/' + psr.name)
    d = check.residuals - base.residuals
    print(f'{nome}: differenza con la sim senza GWB = solo GWB?', np.allclose(d, gwb_only))
    for etichetta, m in [('LOFAR (121-185 MHz)', mask_lofar), ('resto (>1 GHz)', ~mask_lofar)]:
        print(f'   {etichetta:22s}  con GWB {np.std(check.residuals[m]):.3e}   senza {np.std(base.residuals[m]):.3e}')